# Practical Pairwise Alignment Exercises (Biopython, BLAST, ClustalW)
*Computational Biology & Genomics for Neuroscience*

**Learning Objectives**

- Recognize when pairwise alignment, database search, and progressive alignment support neurogenomics questions.
- Practice Biopython tools for nucleotide and protein alignments in small, readable steps.

**Session Roadmap (~60 min)**

- Setup and dataset tour (10 min)
- Pairwise alignment with Biopython (20 min)
- BLAST workflows and result parsing (15 min)

## Context

Brain-Derived Neurotrophic Factor (BDNF) is a key regulator of synaptic plasticity.
We will compare coding sequences from human, mouse, rat, and zebrafish to see how conserved the mature protein is across vertebrates.
To keep the focus on analysis (not downloading), the coding sequences were pre-fetched from Ensembl and saved in this repository as a FASTA file.

## 0. Setup

- These notebooks run inside the course conda environment.
- If you are working elsewhere, install the packages listed below before moving on.

You may have to install a new python package , [biopython](https://biopython.org/). Biopython is a set of freely available tools for biological computation including functionalities for file format parsing, sequence alignment, protein structure analysis, and more.

In [ ]:
# Remove the comment if you need to install the packages in a fresh environment
!pip install biopython

### Imports used throughout the notebook

In [ ]:
from pathlib import Path

import pandas as pd

# The 'biopython' module is actually imported as 'Bio'. We're using this syntax to _only_ import the necessary components, so as not to clutter the namespace with functions/methods we don't need.
from Bio import SeqIO, pairwise2
from Bio.Align import PairwiseAligner, substitution_matrices

## 1. Load BDNF coding sequences from disk

All sequences are stored in `data/pairwise_alignment/bdnf_cds.fasta`.
Each FASTA record contains the coding DNA sequence (CDS) for one species.

Here we will import and then use the [SeqIO](https://biopython.org/wiki/SeqIO) submodule from Biopython to parse the FASTA file and extract the sequences.

In [ ]:
# Read the cached BDNF coding sequences from the FASTA file
from pathlib import Path
from Bio import SeqIO

notebook_folder = Path.cwd()
data_folder = notebook_folder / "data" / "pairwise_alignment"
fasta_path = data_folder / "bdnf_cds.fasta"

# SeqIO.parse reads each FASTA entry as a SeqRecord object
bdnf_records = list(SeqIO.parse(fasta_path, format="fasta"))
print(f"Found {len(bdnf_records)} sequences in {fasta_path.relative_to(notebook_folder)}")

for record in bdnf_records:
    print(f"- {record.id} ({len(record.seq)} nucleotides)")


### Take a quick look at each sequence

We will place basic information in a `pandas` table so it is easy to scan.

In [ ]:
rows = []
for record in bdnf_records:
    rows.append({
        "record_id": record.id,
        "description": record.description,
        "length_nt": len(record.seq),
        "sequence": str(record.seq),
    })

summary_df = pd.DataFrame(rows)
summary_df

### Exercise 1 · transcript sanity checks (10 min)

1. Compute GC content for each sequence and add it as a new column in `summary_df`.
2. Check whether each sequence begins with the start codon `ATG`.
3. Check whether each sequence ends with one of the stop codons: `TAA`, `TAG`, or `TGA`.
4. Print a short note if anything looks unusual, and suggest a biological explanation (e.g., alternative start site)

## 2. Pairwise alignment with Biopython (20 min)

We will start with a global alignment (Needleman–Wunsch) between the human and mouse coding sequences.
`pairwise2.align.globalms` uses four scores: match, mismatch, gap open, and gap extend.

In [ ]:
human_seq = str(bdnf_records[0].seq)
mouse_seq = str(bdnf_records[1].seq)

alignment_list = pairwise2.align.globalms(human_seq, mouse_seq, 2, -1, -8, -2, penalize_end_gaps=False)
print(f"pairwise2 returned {len(alignment_list)} alignments. Showing the best score below:")
print(pairwise2.format_alignment(*alignment_list[0]))

The alignment shows the two sequences with matches (`|`) and gaps (`-`).
Look for concentrated gaps: they often reflect alternative exons or indels introduced during evolution.

### Exercise 2 · try new scoring settings (10 min)

- Re-run the global alignment with at least two new parameter sets.
- Suggested changes: increase the gap-open penalty (for example `-12`), or reward transitions (A↔G, C↔T) slightly more than other mismatches.
- Compare the printed alignments and describe which setting best preserves the functional region you expect to be conserved.

### Using the `PairwiseAligner` class

`PairwiseAligner` is a newer interface that exposes more options and metadata.
Below we run a global alignment between mouse and rat coding sequences.

In [ ]:
aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -8
aligner.extend_gap_score = -2

rat_seq = str(bdnf_records[2].seq)
alignments_mouse_rat = aligner.align(mouse_seq, rat_seq)
print(f"Top alignment score: {alignments_mouse_rat.score}")
print(alignments_mouse_rat[0])

### Exercise 3 · local alignment for motif discovery (5 min)

Switch the aligner to `mode = "local"` and align the human sequence against the zebrafish sequence.
Report the highest scoring local region and relate it to the pre-pro-BDNF structure (prodomain vs. mature domain).

In [ ]:
# TODO: change aligner.mode to "local", run the alignment, and inspect the top fragment.


### Translate each coding sequence to protein

The coding DNA sequences translate to the BDNF precursor protein. Comparing proteins helps us see whether nucleotide differences change amino acids.

In [ ]:
protein_rows = []
for record in bdnf_records:
    protein_seq = record.seq.translate(to_stop=True)
    protein_rows.append({
        "record_id": record.id,
        "amino_acids": str(protein_seq),
        "length_aa": len(protein_seq),
    })

protein_df = pd.DataFrame(protein_rows)
protein_df

## 3. BLAST workflows (15 min)

BLAST extends our comparison to larger databases. Here we build a FASTA query file for the mouse sequence and prepare a remote BLAST call. This will require an internet connection to work.

If you plan to run BLAST set `RUN_REMOTE_BLAST = True`. Keep it `False` when internet access is not available.

In [ ]:
blast_folder = data_folder / "blast"
blast_folder.mkdir(exist_ok=True)

mouse_record = bdnf_records[1]
query_path = blast_folder / "mouse_bdnf_query.fa"
SeqIO.write(mouse_record, query_path, "fasta")
print(f"Saved query FASTA to {query_path.relative_to(notebook_folder)}")

In [ ]:
from Bio.Blast import NCBIWWW

RUN_REMOTE_BLAST = False  # Change to True only if you have an active internet connection..
blast_xml_path = blast_folder / "mouse_vs_nt.xml"

if RUN_REMOTE_BLAST:
    print("Submitting BLASTn search against NCBI nt (this can take 1-2 minutes)...")
    with NCBIWWW.qblast(
        program="blastn",
        database="nt",
        sequence=str(mouse_record.seq),
        entrez_query="brain",
        expect=1e-10,
        hitlist_size=50,
        format_type="XML",
    ) as blast_handle:
        blast_output = blast_handle.read()
    blast_xml_path.write_text(blast_output)
    print(f"Saved BLAST XML output to {blast_xml_path.relative_to(notebook_folder)}")
else:
    print("Remote BLAST is skipped. Turn it on when you have network access and an NCBI account.")

### Exercise 4 · parse BLAST results (10 min)

Once `mouse_vs_nt.xml` exists, use `Bio.SearchIO` to summarize the top hits.
Include accession, description, percent identity, alignment length, and E-value.
Highlight at least one non-mammalian hit and comment on its relevance to BDNF biology.

In [ ]:
from Bio import SearchIO

if blast_xml_path.exists():
    blast_qresult = SearchIO.read(blast_xml_path, "blast-xml")
    top_rows = []
    for hit in blast_qresult.hits[:10]:
        best_hsp = hit.hsps[0]
        span = best_hsp.aln_span if best_hsp.aln_span else 1
        percent_identity = (best_hsp.ident_num / span) * 100
        top_rows.append({
            "hit_id": hit.id,
            "description": hit.description,
            "percent_identity": round(percent_identity, 2),
            "alignment_length": best_hsp.aln_span,
            "evalue": best_hsp.evalue,
        })
    top_rows_table = pd.DataFrame(top_rows)
else:
    print("No BLAST XML file yet. Run the cell above after enabling RUN_REMOTE_BLAST.")

top_rows_table


## 4. ClustalW for pairwise distances (15 min)

ClustalW builds a multiple alignment and guide tree.
From the resulting alignment we can compute pairwise identity values to see which species are closest.

In [ ]:
# Uncomment the following line to install ClustalW, you can recomment it after you have installed clustalw
#!conda install -y -c bioconda clustalw

In [ ]:
import shutil

clustal_path = shutil.which("clustalw") or shutil.which("clustalw2")
if clustal_path:
    print(f"ClustalW found at: {clustal_path}")
else:
    print("ClustalW not detected. Install it (for example: conda install clustalw) before running the next cell.")

In [ ]:
clustal_alignment_path = data_folder / "bdnf_cds.aln"

if clustal_path:
    import subprocess

    command = [
        clustal_path,
        f"-INFILE={fasta_path}",
        f"-OUTFILE={clustal_alignment_path}",
        "-OUTPUT=CLUSTAL",
        "-OUTORDER=INPUT",
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode == 0:
        print("ClustalW alignment complete.")
        print(result.stdout)
    else:
        print("ClustalW reported an error:")
        print(result.stderr)
else:
    print("Skipping ClustalW run because the executable is missing.")


### Exercise 5 · build a pairwise identity heatmap (10 min)

1. Read the `.aln` file with `Bio.AlignIO.read`.
2. Compute the fraction of identical characters for every pair of sequences.
3. Plot the matrix with `seaborn.heatmap` to visualize conservation.
4. Identify the most diverged pair and connect it to vertebrate phylogeny.

In [ ]:
from Bio import AlignIO
import numpy as np

if clustal_alignment_path.exists():
    alignment = AlignIO.read(clustal_alignment_path, "clustal")
    num_records = len(alignment)
    labels = [record.id for record in alignment]
    identity_matrix = np.zeros((num_records, num_records))

    for i in range(num_records):
        for j in range(num_records):
            matches = 0
            total = alignment.get_alignment_length()
            for a_base, b_base in zip(str(alignment[i].seq), str(alignment[j].seq)):
                if a_base == b_base:
                    matches += 1
            identity_matrix[i, j] = matches / total

    identity_df = pd.DataFrame(identity_matrix, index=labels, columns=labels)
else:
    print("ClustalW alignment not found. Run the previous cell after installing ClustalW.")


In [ ]:
identity_df


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(identity_df, annot=True, fmt=".2f", cmap="viridis")
plt.title("Pairwise identity from the BDNF CDS alignment")